# Ark+ pretraining on 12 2D MedMNIST datasets

Drives the real `Ark_Plus/Pretraining` code (`engine.py`/`trainer.py`/`models.py`/`dataloader.py`, unmodified except for the compat shims and crash-proof resume documented in `DEVIATIONS.md`) against the MedMNIST data layer from `medmnist_dataloader.py`. Run `verify_env.py` first if you haven't.

In [ ]:
import sys, os
sys.argv = ['pretrain_medmnist2d']  # keep optparse away from Jupyter's own -f <kernel.json>

from main_ark import get_args_parser
from dataloader import dict_dataloarder, build_transform_classification
from medmnist_dataloader import MEDMNIST_DATALOADER_DICT, MEDMNIST_2D_KEYS
from utils import get_config
from engine import omni_engine

dict_dataloarder.update(MEDMNIST_DATALOADER_DICT)  # register the 12 MedMNIST classes


## Configure the run

`swin_tiny` + 112px + batch 32 is sized for an 8GB card with ~2-6GB free (see `DEVIATIONS.md`) -- **run `nvidia-smi` right before this and reduce `batch_size` if free VRAM is lower than it was at planning time.**

In [ ]:
args = get_args_parser(argv=[])

args.model_name = "swin_tiny"
args.dataset_list = MEDMNIST_2D_KEYS
args.crop_size = 112
args.resize = 128
args.batch_size = 32
args.workers = 4
args.pretrain_epochs = 15          # capped for a Wed deadline run, not the paper default (25) -- see DEVIATIONS.md
args.momentum_teacher = 0.9
args.ema_mode = "epoch"
args.exp_name = "medmnist2d_swintiny"
args.projector_features = 512
args.use_mlp = False
args.pretrained_weights = None      # ImageNet init applied separately below, not via this path
args.opt = "momentum"
args.lr = 1e-2
args.momentum = 0.9
args.weight_decay = 1e-4
args.resume = False                 # original epoch-only resume path -- off, we use crash_proof_resume
args.crash_proof_resume = True      # THE crash-proof, per-dataset resume path (Task 7)
args.use_amp = True                 # DEVIATION, required for 8GB VRAM -- see DEVIATIONS.md
args.reinit_heads = False

print(args)


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free",
                       "--format=csv"], capture_output=True, text=True).stdout)


## Build datasets and model, then hand off to the real `omni_engine` loop

This mirrors `main_ark.py`'s `main()` exactly (same `dict_dataloarder[dataset](...)` construction pattern), just pointed at `datasets_config_medmnist.yaml` instead of `datasets_config.yaml`.

In [ ]:
exp_name = args.model_name + "_" + args.exp_name
model_path = os.path.join("./Models", exp_name)
output_path = os.path.join("./Outputs", exp_name)

datasets_config = get_config('datasets_config_medmnist.yaml')
for dataset in args.dataset_list:
    assert dataset in datasets_config, f"{dataset} missing from datasets_config_medmnist.yaml"

dataset_train_list, dataset_val_list, dataset_test_list = [], [], []
for dataset in args.dataset_list:
    dataset_train_list.append(
        dict_dataloarder[dataset](images_path=datasets_config[dataset]['data_dir'],
                                   file_path=datasets_config[dataset]['train_list'],
                                   crop_size=args.crop_size, resize=args.resize, augment=None))
    dataset_val_list.append(
        dict_dataloarder[dataset](images_path=datasets_config[dataset]['data_dir'],
                                   file_path=datasets_config[dataset]['val_list'],
                                   crop_size=args.crop_size, resize=args.resize,
                                   augment=build_transform_classification(
                                       normalize=args.normalization, crop_size=args.crop_size,
                                       resize=args.resize, mode="valid")))
    dataset_test_list.append(
        dict_dataloarder[dataset](images_path=datasets_config[dataset]['data_dir'],
                                   file_path=datasets_config[dataset]['test_list'],
                                   crop_size=args.crop_size, resize=args.resize,
                                   augment=build_transform_classification(
                                       normalize=args.normalization, crop_size=args.crop_size,
                                       resize=args.resize, mode="test",
                                       test_augment=args.test_augment)))
    print(f"  {dataset}: train={len(dataset_train_list[-1])} "
          f"val={len(dataset_val_list[-1])} test={len(dataset_test_list[-1])}")


In [ ]:
omni_engine(args, model_path, output_path, args.dataset_list, datasets_config,
            dataset_train_list, dataset_val_list, dataset_test_list)


## If this cell was interrupted (Ctrl+C or a crash)

Just re-run the two cells above (config, then `omni_engine(...)`) — `args.crash_proof_resume = True` means it finds `Models/<exp_name>/.../*_atomic_latest.pth` and resumes mid-cycle. See Task 11 for how to verify this.